In [2]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from keras import models,layers
from tensorflow.keras.applications import MobileNetV3Small
from tensorflow.keras.applications.mobilenet_v3 import preprocess_input
from tensorflow.keras.callbacks import ReduceLROnPlateau
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import classification_report
import matplotlib.pyplot as plt

In [ ]:
data = np.load("")

X_train = data["X_train"]
y_train = data["y_train"]

In [ ]:
import numpy as np

data = np.load("")

X_test = data["X_test"]
y_test = data["y_test"]

In [6]:
np.unique(y_train)

array([0, 1, 2, 3, 4, 5, 6, 7])

In [7]:
np.unique(y_test)

array([0, 1, 2, 3, 4, 5, 6, 7], dtype=int32)

In [8]:
X_train_preprocessed = preprocess_input(X_train.copy())
X_test_preprocessed = preprocess_input(X_test.copy())

In [9]:
num_classes = 8
base_model = MobileNetV3Small(
    weights="imagenet",
    include_top=False,
    input_shape=(224,224, 3),
)

2026-10-06 11:31:42.485948: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


In [10]:
base_model.trainable = False

In [11]:
model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.5),
    layers.Dense(num_classes, activation="softmax")
])


In [12]:
reduce_lr = ReduceLROnPlateau(
    monitor='val_accuracy', factor=0.5, patience=3, min_lr=1e-10, verbose=1
)
early_stop = EarlyStopping(
    monitor='val_accuracy', patience=5, restore_best_weights=True,verbose=1
)

In [13]:
def train_evaluate_model(
    optimizer,
    loss_function,
    X_train,
    y_train,   
    X_test,
    y_test,
    epochs,
    callbacks   
    ):
    model.compile(
    optimizer=optimizer,
    loss=loss_function,
    metrics=["accuracy"]
    )
    history = model.fit(
    X_train,
    y_train,
    epochs=epochs,
    validation_data=(X_test, y_test),    
    callbacks=callbacks,
    )
            
    model.evaluate(X_test,y_test)    
    y_pred = model.predict(X_test)
    y_pred = np.argmax(y_pred, axis=1)
    print(classification_report(y_true=y_test,y_pred=y_pred))
    
    return history

In [14]:
optimizer= tf.keras.optimizers.Adam(1e-3)
callbacks = [reduce_lr,early_stop]
history = train_evaluate_model(
    optimizer = optimizer,
    loss_function= "sparse_categorical_crossentropy",
    X_train = X_train_preprocessed,
    y_train = y_train,
    X_test = X_test_preprocessed,
    y_test = y_test,
    epochs = 20,
    callbacks = callbacks
)

Epoch 1/20
250/250 ━━━━━━━━━━━━━━━━━━━━ 60s 217ms/step - accuracy: 0.6151 - loss: 1.0261 - val_accuracy: 0.8852 - val_loss: 0.3806 - learning_rate: 0.0010
Epoch 2/20
250/250 ━━━━━━━━━━━━━━━━━━━━ 56s 223ms/step - accuracy: 0.7736 - loss: 0.6168 - val_accuracy: 0.9031 - val_loss: 0.3063 - learning_rate: 0.0010
Epoch 3/20
250/250 ━━━━━━━━━━━━━━━━━━━━ 52s 206ms/step - accuracy: 0.8121 - loss: 0.5217 - val_accuracy: 0.9005 - val_loss: 0.2821 - learning_rate: 0.0010
Epoch 4/20
250/250 ━━━━━━━━━━━━━━━━━━━━ 52s 207ms/step - accuracy: 0.8300 - loss: 0.4667 - val_accuracy: 0.9031 - val_loss: 0.2750 - learning_rate: 0.0010
Epoch 5/20
250/250 ━━━━━━━━━━━━━━━━━━━━ 50s 201ms/step - accuracy: 0.8450 - loss: 0.4219 - val_accuracy: 0.9184 - val_loss: 0.2450 - learning_rate: 0.0010
Epoch 6/20
250/250 ━━━━━━━━━━━━━━━━━━━━ 52s 206ms/step - accuracy: 0.8643 - loss: 0.3741 - val_accuracy: 0.9005 - val_loss: 0.2749 - learning_rate: 0.0010
Epoch 7/20
250/250 ━━━━━━━━━━━━━━━━━━━━ 50s 199ms/step - accuracy: 0.8

In [ ]:
import json

with open("nails_disease_multi_class_classifier_model_history.json", "w") as f:
    json.dump(history.history, f)

In [4]:
with open("nails_disease_multi_class_classifier_model_history.json", "r") as f:
    history_dict = json.load(f)
history_dict

In [20]:
model.save("nails_disease_multi_class_classification_model.keras")

In [8]:
del X_train,X_test
import gc
gc.collect()

44